## Подготовка окружения

Сначала выбери kernel `Python (RAG venv)`, если ты создал отдельное виртуальное окружение. Затем один раз выполни следующую ячейку установки зависимостей.


In [ ]:
import subprocess
import sys

packages = [
    "langchain",
    "langchain-community",
    "langchain-text-splitters",
    "pypdf",
    "langchain-ollama",
    "chromadb",
]

subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "pip"])
subprocess.check_call([sys.executable, "-m", "pip", "install", *packages])
print("Готово: зависимости установлены в", sys.executable)


In [ ]:
import sys
from pathlib import Path

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

PDF_PATH = Path("Адреса и аудитории.pdf")
print("Python:", sys.executable)
print("PDF найден:", PDF_PATH.exists(), "->", PDF_PATH.resolve())


# Как работает RAG

## **1. Индексация**

Обычно индексация работает следующим образом:

1. **Загрузка**: сначала нам нужно загрузить наши данные. Это делается с помощью **загрузчиков документов**.
2. **Разделение**: **Разделители текста** разбивают большие документы на более мелкие фрагменты. Это полезно как для индексации данных, так и для передачи их в модель, поскольку большие фрагменты сложнее искать и они не помещаются в конечное контекстное окно модели.
3. **Store**: нам нужно где-то хранить и индексировать наши разбиения, чтобы потом можно было выполнять поиск по ним. Часто для этого используют **VectorStore** и модель **Embeddings**.

### Загрузка документов

- `PyPDFLoader`, наоборот, предназначен специально для PDF-файлов.
Он извлекает текст постранично и возвращает список документов,
где каждая страница – отдельный Document с метаданными страницы.
Это удобнее для RAG, но качество извлечения зависит от структуры PDF:
если PDF – скан или сложное форматирование, текст может извлекаться не идеально.
  

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(str(PDF_PATH))
docs = loader.load()

### **Разделение документов**

Загруженный нами документ содержит ~10 000 символов. Контекстное окно современных моделей способно переварить такое количество входных данных. Но что, если у нас будет не 1 документ, а 10, 100, или 1000? Чтобы решить эту проблему, мы разделим **`Document`** на фрагменты для встраивания и векторное хранилище. Это поможет нам извлекать только наиболее важные части поста во время выполнения задачи.

Для этого мы используем **`RecursiveCharacterTextSplitter`**, который будет рекурсивно разбивать документ на небольшие части (чанки) с помощью стандартных разделителей (таких как символы новой строки), до тех пор, пока размер каждой части не будет соответствовать требованиям.

Это рекомендуемый разделитель текста для общих случаев использования текста.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)


### Создаём embeddings

После разбиения текста на чанки нам нужно преобразовать каждый фрагмент в числовое представление – **эмбеддинг**.

Эмбеддинги – это векторы из десятков, сотен или тысяч чисел, и каждый такой вектор – это точка в многомерном пространстве. В этом пространстве смысловые связи отражаются геометрически: похожие тексты оказываются "близко" друг к другу, а непохожие – далеко. Векторное хранилище (например, Chroma, Qdrant или Pinecone) сохраняет эти точки и позволяет быстро находить ближайшие, измеряя расстояние между ними через cosine similarity, dot product или другую метрику. Именно это и делает возможным семантический поиск: модель не ищет совпадение слов – она ищет **смысловую близость в пространстве эмбеддингов**.

Модель для создания эмбеддингов – это специальная нейросеть, которая преобразует текст в такие векторные представления и позволяет выполнять семантический поиск по базе документов.

От выбора embedding-модели зависит качество поиска: чем лучше она понимает язык и контекст, тем точнее система сможет находить нужные фрагменты. Выбор модели определяется языком данных, доступными ресурсами (нужна ли локальная модель или облачная) и требуемым балансом между скоростью, стоимостью и качеством.
![image.png](RAG_files/rag-embedded-image-1.png)

Для нашего примера возьмём локальную какую-нибудь эмбединг модель с Hugging Face и запустим её локально через Ollama. Пример запроса:

Загрузим модель через Ollama: